## Pipeline Script 


Inputs: Group CEST and nmap data output from pyGluCEST as well as demographic data from _________. 
Outputs: Compiled dataframes with GluCEST and nmap data. Trimmed based on number of people with sufficient data.

    Trimmed subject-wise dfs: e.g., cestmat (outpath + 'trimmed_cestmat' + dataset + atlas + '.csv')
    Long form dfs: e.g., long_df (outpath + 'longform_grpdf' + dataset + '_' + atlas + '.csv')
         Also have version with standard nmap values
    Mean dfs: e.g., grouped_df (outpath + 'means_' + dataset + '_' + atlas + '.csv')


### Import Packages

In [1]:
import os
import glob
import numpy as np
import pandas as pd
#import network_fcon as fc
import scipy as sp
from scipy.stats import pearsonr
from scipy.stats import linregress
import seaborn as sns
import matplotlib.pyplot as plt
import re
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012

In [2]:
lobe_dict = {
    "caudalanteriorcingulate":"frontal",
    "cuneus":"occipital",
    "frontalpole":"frontal",
    "isthmuscingulate":"parietal",
    "lingual":"occipital",
    "paracentral":"frontal",
    "posteriorcingulate":"parietal",
    "precuneus":"parietal",
    "rostralanteriorcingulate":"frontal",
    "superiorfrontal":"frontal"
}
    
def add_lobe(df):
    if "lobe" in df.columns:
        return df
    else:
        parcel_part = df['Parcel'].str.extract(r'(.*?)_')
        df["Lobe"] = [lobe_dict[key] for key in parcel_part[0] if key in lobe_dict]
        return df


### Define paths and variables

In [2]:
# Set variables
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500'
nmaps = ["NMDA", "mGluR5", "GABA"]
maps = ["cest", "NMDA", "mGluR5", "GABA"]
normalize_cest = True

# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/" + dataset
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
os.makedirs(os.path.join(outpath), exist_ok=True)

# Read in data
cestmat = pd.read_csv(inpath + "/all_subs_GluCEST_" + atlas + "_UNI.csv", sep=',')
NMDAmat = pd.read_csv(inpath + "/all_subs_NMDA_normalized_" + atlas + "_UNI.csv", sep=',')
mGluR5mat = pd.read_csv(inpath + "/all_subs_mGluR5_normalized_" + atlas + "_UNI.csv", sep=',')
GABAmat = pd.read_csv(inpath + "/all_subs_GABA_normalized_" + atlas + "_UNI.csv", sep=',')

# Set indices and correct column names
cestmat.set_index('Subject', inplace = True)
NMDAmat.set_index('Subject', inplace = True)
GABAmat.set_index('Subject', inplace = True)
mGluR5mat.set_index('Subject', inplace = True)
dfs = [cestmat, NMDAmat, mGluR5mat, GABAmat]

# Load in standardized nmap data for alternative approach.
receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_cammoun2012_scale500.csv", sep=',')


In [3]:
# Remove Subcortical ROIs
cam = fetch_cammoun2012()
info = pd.read_csv(cam['info'], sep=',')
info = info[info['scale']== 'scale500']
#labels = info['label'][info['structure']=='cortex'].values
#labels = info['label'][info['structure']=='cortex'].values + hemisphere
hemisphere = info['hemisphere'][info['structure']=='subcortex'].values
subcortex = info['label'][info['structure']=='subcortex'].values + hemisphere
print(subcortex.shape)

(15,)


In [4]:
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
#print(cestmat)

## Trim Data

In [5]:
# ID parcels with < 20 voxels* 
for i, col in enumerate(cestmat.columns):
    if 'NZcount' in col:
        # Set mean col to nan
        mean_col = cestmat.columns[i - 1]
        sigma_col = cestmat.columns[i + 1]
        cestmat[mean_col] = np.where(cestmat[col] < 20, np.nan, cestmat[mean_col])
        cestmat[sigma_col] = np.where(cestmat[col] < 20, np.nan, cestmat[sigma_col])
        cestmat[col] = np.where(cestmat[col] < 20, np.nan, cestmat[col])       
columns = cestmat.columns[cestmat.notnull().sum() > len(cestmat)*.75]
print(cestmat.shape)

# Trim all dfs based on column filter
cestmat= cestmat[columns]
NMDAmat= NMDAmat[columns]
GABAmat= GABAmat[columns]
mGluR5mat= mGluR5mat[columns]
print(cestmat.shape)

# ID subjects missing >65% of remaining GluCEST parcels
sparse_subjs = cestmat[cestmat.isna().sum(axis=1) > cestmat.shape[1] * 0.65].index

# Trim all dfs based on row filter
cestmat = cestmat.drop(index=sparse_subjs)
NMDAmat = NMDAmat.drop(index=sparse_subjs)
GABAmat = GABAmat.drop(index=sparse_subjs)
mGluR5mat = mGluR5mat.drop(index=sparse_subjs)
print(cestmat.shape)

# Remove subcortical ROIs
if atlas == 'atl-Cammoun2012_res-500':
    cestmat = cestmat.loc[:, ~cestmat.columns.str.contains('|'.join(subcortex), na=False)]
    NMDAmat = NMDAmat.loc[:, ~NMDAmat.columns.str.contains('|'.join(subcortex), na=False)]
    GABAmat = GABAmat.loc[:, ~GABAmat.columns.str.contains('|'.join(subcortex), na=False)]
    mGluR5mat = mGluR5mat.loc[:, ~mGluR5mat.columns.str.contains('|'.join(subcortex), na=False)]
print(cestmat.shape)

# Temporary: Remove mysterious zeros in nmap dataframes
dfs = [NMDAmat, mGluR5mat, GABAmat]
for i in range(len(dfs)):
    df = dfs[i]
    df.replace(0, np.nan, inplace=True)

# Save trimmed dfs
cestmat.to_csv(outpath + '/trimmed_cestmat' + dataset + atlas + '.csv', index=True)
NMDAmat.to_csv(outpath + '/trimmed_NMDAmat' + dataset + atlas + '.csv', index=True)
GABAmat.to_csv(outpath + '/trimmed_GABAmat' + dataset + atlas + '.csv', index=True)
mGluR5mat.to_csv(outpath + '/trimmed_mGluR5mat' + dataset + atlas + '.csv', index=True)

(182, 725)
(182, 167)
(176, 167)
(176, 158)


## Add Cortical Lobe Labels

In [6]:
cestmat.head()

,Unnamed: 0,frontalpole_1R NZMean,frontalpole_1R NZcount,frontalpole_1R NZSigma,superiorfrontal_40R NZMean,superiorfrontal_40R NZcount,superiorfrontal_40R NZSigma,superiorfrontal_7R NZMean,superiorfrontal_7R NZcount,superiorfrontal_7R NZSigma,...,cuneus_3R NZMean,cuneus_3R NZcount,cuneus_3R NZSigma,cuneus_5R NZMean,cuneus_5R NZcount,cuneus_5R NZSigma,precuneus_15R NZMean,precuneus_15R NZcount,precuneus_15R NZSigma,group
Subject,,,,,,,,,,,,,,,,,,,,,
100522_12003,0,5.533474,32.0,2.188992,4.299698,49.0,2.442113,5.721321,45.0,2.229686,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TD/NC
100522_12371,1,5.459877,28.0,2.447069,5.136139,39.0,2.566857,6.545109,48.0,2.199931,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TD/NC
100522_12783,2,4.864008,31.0,1.728823,6.269497,51.0,2.130490,6.568427,62.0,1.743864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TD/NC
102041_12037,3,7.210485,69.0,2.443378,7.443285,66.0,1.365854,7.582340,60.0,1.850772,...,7.258384,41.0,2.249775,8.277832,30.0,1.703806,NaN,NaN,NaN,PRO/CHR
102041_12500,4,9.612453,71.0,2.522940,7.381907,63.0,1.860883,8.027369,56.0,1.492858,...,9.191903,42.0,1.569125,9.379838,34.0,1.157970,NaN,NaN,NaN,PRO/CHR


## Normalize GluCEST

In [7]:
# Step 1: Select columns that contain 'NZMean'
if normalize_cest:
    nzmean_columns = [col for col in cestmat.columns if 'NZMean' in col]
    
    # Step 2: Calculate mean and std deviation for each subject (row-wise) across selected columns
    cestmat['Subject_Avg_NZMean'] = cestmat[nzmean_columns].mean(axis=1)
    cestmat['Subject_Std_NZMean'] = cestmat[nzmean_columns].std(axis=1)
    
    # Step 3: Calculate z-scores for all selected columns at once and store them in a new dataframe
    zscore_df = (cestmat[nzmean_columns].sub(cestmat['Subject_Avg_NZMean'], axis=0)
                 .div(cestmat['Subject_Std_NZMean'], axis=0))

    # Step 4: Concatenate the z-scores dataframe to the original cestmat dataframe
    cestmat = pd.concat([cestmat['group'], zscore_df], axis=1)
    cestmat.to_csv(outpath + '/grp_df_means_std_normalized_' + dataset + '_' + atlas + '.csv', index=False)


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_45761/2696464397.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cestmat['Subject_Avg_NZMean'] = cestmat[nzmean_columns].mean(axis=1)
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_45761/2696464397.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cestmat['Subject_Std_NZMean'] = cestmat[nzmean_columns].std(axis=1)


In [8]:
cestmat.head()

,group,frontalpole_1R NZMean,superiorfrontal_40R NZMean,superiorfrontal_7R NZMean,superiorfrontal_23R NZMean,superiorfrontal_10R NZMean,superiorfrontal_39R NZMean,superiorfrontal_29R NZMean,superiorfrontal_1R NZMean,superiorfrontal_30R NZMean,...,paracentral_11R NZMean,precuneus_7R NZMean,precuneus_14R NZMean,precuneus_11R NZMean,precuneus_8R NZMean,precuneus_18R NZMean,precuneus_13R NZMean,cuneus_3R NZMean,cuneus_5R NZMean,precuneus_15R NZMean
Subject,,,,,,,,,,,,,,,,,,,,,
100522_12003,TD/NC,-1.573371,-2.511252,-1.430575,-0.694887,-1.330532,-1.444310,-0.465021,-1.045952,-0.017369,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12371,TD/NC,-2.009694,-2.268959,-1.140590,-0.077094,-1.079176,-1.159484,0.090207,-0.169905,0.008017,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100522_12783,TD/NC,-2.328360,-1.216252,-0.979721,0.337610,-0.339099,-1.239030,-0.354976,0.366586,-0.664460,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
102041_12037,PRO/CHR,-0.686951,-0.495555,-0.381232,-0.279610,-0.312438,-0.953016,-1.139706,-0.405014,-0.511538,...,0.784675,NaN,0.122211,0.234537,NaN,-0.786964,-0.379243,-0.647571,0.190564,NaN
102041_12500,PRO/CHR,2.060564,-0.101545,0.524113,0.030853,-0.082949,0.184125,-0.712321,0.192955,-0.690277,...,-0.960837,NaN,0.274427,-0.385214,-0.418084,0.905397,-0.489707,1.652917,1.835086,NaN


## Make classic grp_df

In [9]:
# First, add datatype to column names so can be distinguished later.
cestmat2=cestmat.copy()
NMDAmat2=NMDAmat.copy()
GABAmat2=GABAmat.copy()
mGluR5mat2=mGluR5mat.copy()
cestmat2.columns = [f"GluCEST_{col}" if "NZ" in col else col for col in cestmat2.columns]
NMDAmat2.columns = [f"NMDA_{col}" if "NZ" in col else col for col in NMDAmat2.columns]
GABAmat2.columns = [f"GABA_{col}" if "NZ" in col else col for col in GABAmat2.columns]
mGluR5mat2.columns = [f"mGluR5_{col}" if "NZ" in col else col for col in mGluR5mat2.columns]

# Align dataframes by "Subject" index and concatenate along columns
grp_df = cestmat2.join(NMDAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(GABAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(mGluR5mat2.filter(like='NZ'), how='left')

# Save grpd_df
grp_df.to_csv(outpath + '/grp_df_' + dataset + atlas + '.csv', index=True)
#print(cestmat)

In [10]:
#print(NMDAmat)

## Make longform group df

In [11]:
# Make longform group df
# Get list of parcel names
parcels = cestmat.filter(like="NZMean").columns.tolist()

# Melt cestmat to get Glu data in long format
cestlong = cestmat.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                      var_name='Parcel', value_name='GluCEST')

# Melt nmap data. Fix!! turn into loop later.
NMDAlong = NMDAmat.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                       var_name='Parcel', value_name='NMDA')
GABAlong = GABAmat.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                       var_name='Parcel', value_name='GABA')
mGluR5long = mGluR5mat.reset_index().melt(id_vars='Subject', value_vars=parcels, 
                                       var_name='Parcel', value_name='mGluR5')

# Merge the long-form dataframes based on Subject and Parcel
long_df = pd.merge(cestlong, NMDAlong,  on=['Subject', 'Parcel'])
long_df = pd.merge(long_df, GABAlong,  on=['Subject', 'Parcel'])
long_df = pd.merge(long_df, mGluR5long,  on=['Subject', 'Parcel'])

# Add diagnostic group
diag_df = cestmat['group']
long_df = pd.merge(long_df, diag_df, on='Subject')
long_df['hstatus'] = np.where(long_df['group'].isin(['TD/NC']), 'HC', 'PSY')
#print(long_df)

# Save longformdf
long_df.to_csv(outpath + '/longform_grpdf_' + dataset + atlas + '.csv', index=True)

### Make mean group dfs by diagnosis

In [12]:
# Make mean group dfs by diagnosis
grouped_subj = long_df.groupby(['Parcel', 'hstatus']).agg(
    CEST_avg=('GluCEST', 'mean'),
    NMDA=('NMDA', 'mean'),
    mGluR5=('mGluR5', 'mean'),
    GABA=('GABA', 'mean')
).reset_index()
print(grouped_subj)
# Save
grouped_subj.to_csv(outpath + '/means_subjectnmaps_' + dataset + '_' + atlas + '.csv', index=False)

                                Parcel hstatus  CEST_avg      NMDA    mGluR5  \
0    caudalanteriorcingulate_5R NZMean      HC  0.140048 -1.047351  0.278927   
1    caudalanteriorcingulate_5R NZMean     PSY -0.011085 -1.210817  0.019354   
2                     cuneus_3R NZMean      HC  0.884299  0.750100  1.195524   
3                     cuneus_3R NZMean     PSY  0.305783  0.808305  1.227833   
4                     cuneus_5R NZMean      HC  1.411665  0.029143  0.314619   
..                                 ...     ...       ...       ...       ...   
99           superiorfrontal_7R NZMean     PSY -0.455633  0.716904  1.076864   
100          superiorfrontal_8R NZMean      HC -0.857712  0.380748  0.482610   
101          superiorfrontal_8R NZMean     PSY -0.800876  0.242487  0.391164   
102          superiorfrontal_9R NZMean      HC -0.154002 -0.110998  0.021254   
103          superiorfrontal_9R NZMean     PSY -0.372532 -0.306964 -0.112934   

         GABA  
0   -0.089857  
1   -0.

# REPEAT USING STANDARD NMAPS

In [13]:
"""
from netneurotools.datasets import fetch_cammoun2012

# Import and add parcel labels to standard receptor_df
if atlas == 'schaefer':
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer.labels
    labels = [label.decode('utf-8') for label in labels]
    receptor_df.index = labels
    receptor_df.index.name = 'Parcel'

if atlas == 'cammoun2012':
"""

receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
print(receptor_df)   

# Chop up receptor_df by map
NMDAmat = receptor_df[["Parcel","NMDA"]]
GABAmat = receptor_df[["Parcel","GABA"]]
mGluR5mat = receptor_df[["Parcel","mGluR5"]]

#print(NMDAmat)

                        Parcel      NMDA    mGluR5      GABA
0      lateralorbitofrontal_9R -0.020868  0.164621 -0.150228
1     lateralorbitofrontal_11R  0.366013 -0.085390  0.170483
2      lateralorbitofrontal_5R  1.171849  1.015446  1.317157
3      lateralorbitofrontal_6R  0.715088 -0.055909 -0.026400
4      lateralorbitofrontal_7R -0.125005  0.105069  0.344259
...                        ...       ...       ...       ...
1010                 pallidumL  1.184228 -2.200435 -3.248923
1011            accumbensareaL  0.632115  0.061453 -0.237722
1012              hippocampusL  0.855395 -1.009223 -1.039651
1013                 amygdalaL  0.109529 -0.782412 -0.971487
1014                brainstemL -0.798938 -4.706040 -5.108990

[1015 rows x 4 columns]


### Make classic grp_df

In [14]:
# Transpose receptor maps.
nmda = NMDAmat.T
gaba = GABAmat.T
mglur5 = mGluR5mat.T

print(nmda)
# Keep only parcels contained in cestmat
cestmat_regions = [col.replace(' NZMean', '') for col in cestmat.columns if ' NZMean' in col]
nmda_filtered = nmda[[col for col in nmda.columns if col in cestmat_regions]]
gaba_filtered = gaba[[col for col in gaba.columns if col in cestmat_regions]]
mglur5_filtered = mglur5[[col for col in mglur5.columns if col in cestmat_regions]]

# Filtered columns
nmda_filtered.columns = [f"NMDA_{col}" for col in nmda_filtered.columns]
gaba_filtered.columns = [f"GABA_{col}" for col in gaba_filtered.columns]
mglur5_filtered.columns = [f"mGluR5_{col}" for col in mglur5_filtered.columns]

# Repeat values for length of cestmat
nmda_repeated = pd.concat([nmda_filtered] * len(cestmat), ignore_index=True)
gaba_repeated = pd.concat([gaba_filtered] * len(cestmat), ignore_index=True)
mglur5_repeated = pd.concat([mglur5_filtered] * len(cestmat), ignore_index=True)

print(nmda_repeated)
# Concatenate
grp_df_std = pd.concat([cestmat2, nmda_repeated, gaba_repeated, mglur5_repeated], axis=1)

# Save grp_df_std
grp_df_std.to_csv(outpath + '/grp_df_std' + dataset + atlas + '.csv', index=True)
print(grp_df_std.head())

                           0                         1     \
Parcel  lateralorbitofrontal_9R  lateralorbitofrontal_11R   
NMDA                  -0.020868                  0.366013   

                           2                        3     \
Parcel  lateralorbitofrontal_5R  lateralorbitofrontal_6R   
NMDA                   1.171849                 0.715088   

                           4                         5     \
Parcel  lateralorbitofrontal_7R  lateralorbitofrontal_10R   
NMDA                  -0.125005                  0.039044   

                           6                         7     \
Parcel  lateralorbitofrontal_4R  lateralorbitofrontal_17R   
NMDA                   1.248462                   0.37993   

                           8                         9     ...       1005  \
Parcel  lateralorbitofrontal_8R  lateralorbitofrontal_15R  ...  insula_3L   
NMDA                  -0.529395                  0.130049  ...  -0.166495   

              1006             1007

### Make Longform df

In [15]:
# Make longform df using standardized nmaps values

# Get list of parcel names
parcels = cestmat.filter(like="NZMean").columns.tolist()

# Keep relevant columns from long_df and rename parcels
longdf_cest = long_df[["Subject","Parcel","GluCEST","group","hstatus"]]
longdf_cest = longdf_cest.replace(' NZMean', '', regex=True)

#longdf_cest["Parcel"] = longdf_cest["Parcel"].str.replace(' NZMean', '', regex=False)

# Convert receptor_df from wide to long format for merging
nmda_long = NMDAmat.reset_index().melt(id_vars='Parcel', var_name='Receptor1', value_name='NMDA_standard')
gaba_long = GABAmat.reset_index().melt(id_vars='Parcel', var_name='Receptor2', value_name='GABA_standard')
mglur5_long = mGluR5mat.reset_index().melt(id_vars='Parcel', var_name='Receptor3', value_name='mGluR5_standard')

long_df_std = pd.merge(longdf_cest, NMDAmat, on=["Parcel"])
long_df_std = pd.merge(long_df_std, GABAmat, on = ["Parcel"])
long_df_std = pd.merge(long_df_std, mGluR5mat, on = ["Parcel"])

# Save the longform dataframe to a CSV
long_df_std.to_csv(outpath + '/longform_grpdf_std_' + dataset + '_' + atlas + '.csv', index=False)


### Make mean df by diagnosis

In [37]:
# Make mean group dfs by diagnosis
# Standard nmap data
grouped_std = long_df_std.groupby(['Parcel', 'hstatus']).agg(
    CEST_avg=('GluCEST', 'mean'),
    NMDA=('NMDA', 'mean'),
    mGluR5=('mGluR5', 'mean'),
    GABA=('GABA', 'mean')
).reset_index()
grouped_std = add_lobe(grouped_std)
print(grouped_std)
grouped_std.to_csv(outpath + '/means_std_' + dataset + '_' + atlas + '.csv', index=False)

                          Parcel hstatus  CEST_avg      NMDA    mGluR5  \
0     caudalanteriorcingulate_5R      HC  0.140048 -0.026330  1.229935   
1     caudalanteriorcingulate_5R     PSY -0.011085 -0.026330  1.229935   
2                      cuneus_3R      HC  0.884299 -0.522157  0.053661   
3                      cuneus_3R     PSY  0.305783 -0.522157  0.053661   
4                      cuneus_5R      HC  1.411665  0.046650  0.220775   
5                      cuneus_5R     PSY  0.963478  0.046650  0.220775   
6                 frontalpole_1R      HC -0.475798 -2.137305 -0.306475   
7                 frontalpole_1R     PSY  0.129303 -2.137305 -0.306475   
8            isthmuscingulate_1R      HC  0.265007  0.862035  0.410423   
9            isthmuscingulate_1R     PSY  0.302260  0.862035  0.410423   
10           isthmuscingulate_2R      HC  0.592127  1.319292  1.994640   
11           isthmuscingulate_2R     PSY  0.723430  1.319292  1.994640   
12           isthmuscingulate_3R      

### Data Imputation

In [17]:
# Now, for the long_dfs, impute data based on average across participants for that parcel
# Subject-wise data
merged_df = pd.merge(long_df, grouped_subj[['Parcel', 'hstatus', 'CEST_avg', 'NMDA', 'GABA', 'mGluR5']], on=['Parcel', 'hstatus'], how='left')
#print(merged_df)
merged_df['GluCEST'] = merged_df['GluCEST'].fillna(merged_df['CEST_avg'])
merged_df['NMDA'] = merged_df['NMDA_x'].fillna(merged_df['NMDA_y'])
merged_df['GABA'] = merged_df['GABA_x'].fillna(merged_df['GABA_y'])
merged_df['mGluR5'] = merged_df['mGluR5_x'].fillna(merged_df['mGluR5_y'])
#print(merged_df)
imputed_df = merged_df.drop(columns=['CEST_avg', 'NMDA_x', 'NMDA_y', 'GABA_x', 'GABA_y','mGluR5_x', 'mGluR5_y'])

# Standard data
# First, rename parcels
long_df_std["Parcel"] = long_df_std["Parcel"].str.replace(' NZMean', '', regex=False)
grouped_subj_std = grouped_subj.copy()
grouped_subj_std["Parcel"] = grouped_subj_std["Parcel"].str.replace(' NZMean', '', regex=False)
#print(long_df_std)
merged_df_std = pd.merge(long_df_std, grouped_subj_std[['Parcel', 'hstatus', 'CEST_avg']], on=['Parcel', 'hstatus'], how='left')
merged_df_std['GluCEST'] = merged_df_std ['GluCEST'].fillna(merged_df_std ['CEST_avg'])
#print(merged_df_std)
imputed_df_std = merged_df_std.drop(columns=['CEST_avg'])

imputed_df.to_csv(outpath + '/imputed_long_df_' + dataset + '_' + atlas + '.csv', index=False)
imputed_df_std.to_csv(outpath + '/imputed_long_df_standardnmaps_' + dataset + '_' + atlas + '.csv', index=False)

### Normalize GluCEST values

In [38]:
# Make mean group dfs by diagnosis
# Standard nmap data
grouped_std = long_df_std.groupby(['Parcel', 'hstatus']).agg(
    CEST_avg=('GluCEST', 'mean'),
    NMDA_avg=('NMDA', 'mean'),
    mGluR5_avg=('mGluR5', 'mean'),
    GABA_avg=('GABA', 'mean')
).reset_index()
grouped_std = add_lobe(grouped_std)
print(grouped_std)
grouped_std.to_csv(outpath + '/means_std_normalized_cest_' + dataset + '_' + atlas + '.csv', index=False)

                          Parcel hstatus  CEST_avg  NMDA_avg  mGluR5_avg  \
0     caudalanteriorcingulate_5R      HC  0.140048 -0.026330    1.229935   
1     caudalanteriorcingulate_5R     PSY -0.011085 -0.026330    1.229935   
2                      cuneus_3R      HC  0.884299 -0.522157    0.053661   
3                      cuneus_3R     PSY  0.305783 -0.522157    0.053661   
4                      cuneus_5R      HC  1.411665  0.046650    0.220775   
5                      cuneus_5R     PSY  0.963478  0.046650    0.220775   
6                 frontalpole_1R      HC -0.475798 -2.137305   -0.306475   
7                 frontalpole_1R     PSY  0.129303 -2.137305   -0.306475   
8            isthmuscingulate_1R      HC  0.265007  0.862035    0.410423   
9            isthmuscingulate_1R     PSY  0.302260  0.862035    0.410423   
10           isthmuscingulate_2R      HC  0.592127  1.319292    1.994640   
11           isthmuscingulate_2R     PSY  0.723430  1.319292    1.994640   
12          

In [19]:
# Step 1: Select columns that contain 'NZMean'
nzmean_columns = [col for col in cestmat.columns if 'NZMean' in col]

# Step 2: Calculate mean and std deviation for each subject (row-wise) across selected columns
cestmat['Subject_Avg_NZMean'] = cestmat[nzmean_columns].mean(axis=1)
cestmat['Subject_Std_NZMean'] = cestmat[nzmean_columns].std(axis=1)

# Step 3: Calculate z-scores for all selected columns at once and store them in a new dataframe
zscore_df = (cestmat[nzmean_columns].sub(cestmat['Subject_Avg_NZMean'], axis=0)
             .div(cestmat['Subject_Std_NZMean'], axis=0))

# Rename z-score columns
zscore_df.columns = [col + '_Zscore' for col in zscore_df.columns]
#print(zscore_df.size)
#print(zscore_df)
# Step 4: Concatenate the z-scores dataframe to the original cestmat dataframe
zcestmat = pd.concat([cestmat['group'], zscore_df], axis=1)

zcestmat.to_csv(outpath + '/grp_df_means_std_normalized_' + dataset + '_' + atlas + '.csv', index=False)
#print(zcestmat)


In [40]:
cestmat2=cestmat.copy()
NMDAmat2=NMDAmat.copy()
GABAmat2=GABAmat.copy()
mGluR5mat2=mGluR5mat.copy()
cestmat2.columns = [f"GluCEST_{col}" if "NZ" in col else col for col in cestmat2.columns]
NMDAmat2.columns = [f"NMDA_{col}" if "NZ" in col else col for col in NMDAmat2.columns]
GABAmat2.columns = [f"GABA_{col}" if "NZ" in col else col for col in GABAmat2.columns]
mGluR5mat2.columns = [f"mGluR5_{col}" if "NZ" in col else col for col in mGluR5mat2.columns]

# Align dataframes by "Subject" index and concatenate along columns
grp_df = cestmat2.join(NMDAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(GABAmat2.filter(like='NZ'), how='left')
grp_df = grp_df.join(mGluR5mat2.filter(like='NZ'), how='left')

#grp_df = add_lobe(grp_df)

# Save grp_df
grp_df.to_csv(outpath + '/grp_df_' + dataset + atlas + '.csv', index=True)